In [2]:
"""Minimal I-JEPA (image JEPA) in PyTorch with block masking.

context encoder -> predictor -> predicted target embeddings
target encoder (EMA, no grad) -> target embeddings
loss = smooth L1 in latent space (no pixel reconstruction)
"""
import copy, math, random
import torch, torch.nn as nn, torch.nn.functional as F

IMG, PATCH, DIM, DEPTH, HEADS = 32, 4, 128, 4, 4
G = IMG // PATCH  # grid size (8x8 patches)


class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        h = self.n1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.mlp(self.n2(x))


class Encoder(nn.Module):
    """ViT; optionally keeps only the patches in `idx` (context masking)."""
    def __init__(self, dim=DIM, depth=DEPTH, heads=HEADS):
        super().__init__()
        self.patch = nn.Conv2d(3, dim, PATCH, PATCH)
        self.pos = nn.Parameter(torch.randn(1, G * G, dim) * 0.02)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(depth))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, idx=None):
        x = self.patch(x).flatten(2).transpose(1, 2) + self.pos
        if idx is not None:
            x = x[:, idx]
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


class Predictor(nn.Module):
    """Narrow ViT: context tokens + positional mask tokens -> target embeddings."""
    def __init__(self, dim=DIM, pdim=64, depth=2, heads=4):
        super().__init__()
        self.inp = nn.Linear(dim, pdim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pdim))
        self.pos = nn.Parameter(torch.randn(1, G * G, pdim) * 0.02)
        self.blocks = nn.ModuleList(Block(pdim, heads) for _ in range(depth))
        self.norm = nn.LayerNorm(pdim)
        self.out = nn.Linear(pdim, dim)

    def forward(self, ctx, ctx_idx, tgt_idx):
        B = ctx.size(0)
        x = self.inp(ctx) + self.pos[:, ctx_idx]
        m = (self.mask_token + self.pos[:, tgt_idx]).expand(B, -1, -1)
        x = torch.cat([x, m], 1)
        for b in self.blocks:
            x = b(x)
        return self.out(self.norm(x[:, -len(tgt_idx):]))


def sample_block(scale, aspect=(0.75, 1.5)):
    area = scale * G * G
    ar = random.uniform(*aspect)
    h = max(1, min(G, round(math.sqrt(area / ar))))
    w = max(1, min(G, round(math.sqrt(area * ar))))
    t, l = random.randint(0, G - h), random.randint(0, G - w)
    return {r * G + c for r in range(t, t + h) for c in range(l, l + w)}


def sample_masks(n_targets=4):
    """Shared across the batch so tensor shapes match. Context excludes targets."""
    while True:
        tgt = set().union(*[sample_block(random.uniform(0.15, 0.2)) for _ in range(n_targets)])
        ctx = sample_block(0.85, (1.0, 1.0)) - tgt
        if len(ctx) >= 8 and tgt:
            return sorted(ctx), sorted(tgt)


class JEPA(nn.Module):
    def __init__(self):
        super().__init__()
        self.ctx_enc = Encoder()
        self.predictor = Predictor()
        self.tgt_enc = copy.deepcopy(self.ctx_enc)
        for p in self.tgt_enc.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def ema_update(self, m):
        for pt, pc in zip(self.tgt_enc.parameters(), self.ctx_enc.parameters()):
            pt.mul_(m).add_(pc.detach(), alpha=1 - m)

    def forward(self, x):
        ctx_idx, tgt_idx = sample_masks()
        ci = torch.tensor(ctx_idx, device=x.device)
        ti = torch.tensor(tgt_idx, device=x.device)
        with torch.no_grad():  # targets: full image through EMA encoder
            target = self.tgt_enc(x)[:, ti]
            target = F.layer_norm(target, target.shape[-1:])
        pred = self.predictor(self.ctx_enc(x, ci), ci, ti)
        return F.smooth_l1_loss(pred, target), pred


def synthetic_batch(B, device):
    """Toy images: a random colored square and circle on a noisy background."""
    x = 0.1 * torch.randn(B, 3, IMG, IMG, device=device)
    yy, xx = torch.meshgrid(torch.arange(IMG), torch.arange(IMG), indexing="ij")
    yy, xx = yy.to(device), xx.to(device)
    for i in range(B):
        t, l = random.randint(0, 16), random.randint(0, 16)
        x[i, :, t:t + 12, l:l + 12] += torch.rand(3, 1, 1, device=device)
        cy, cx, r = random.randint(8, 24), random.randint(8, 24), random.randint(3, 7)
        mask = ((yy - cy) ** 2 + (xx - cx) ** 2) < r * r
        x[i][:, mask] += torch.rand(3, 1, device=device)
    return x


if __name__ == "__main__":
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = JEPA().to(dev)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.05)
    steps, m0 = 1000, 0.996

    for step in range(steps):
        x = synthetic_batch(64, dev)
        loss, pred = model(x)
        opt.zero_grad(); loss.backward(); opt.step()
        model.ema_update(m0 + (1 - m0) * step / steps)  # momentum -> 1
        if step % 50 == 0:
            # collapse monitor: std of predictions across the batch (should stay > 0)
            print(f"step {step:4d}  loss {loss.item():.4f}  pred_std {pred.std(0).mean().item():.4f}")

    # Downstream use: pooled target-encoder features
    with torch.no_grad():
        feats = model.tgt_enc(synthetic_batch(8, dev)).mean(1)
    print("features:", tuple(feats.shape))

step    0  loss 0.4971  pred_std 0.1350
step   50  loss 0.0159  pred_std 0.0009
step  100  loss 0.0047  pred_std 0.0008
step  150  loss 0.0045  pred_std 0.0008
step  200  loss 0.0034  pred_std 0.0007
step  250  loss 0.0031  pred_std 0.0007
step  300  loss 0.0024  pred_std 0.0007
step  350  loss 0.0023  pred_std 0.0007
step  400  loss 0.0020  pred_std 0.0007
step  450  loss 0.0020  pred_std 0.0008
step  500  loss 0.0018  pred_std 0.0008
step  550  loss 0.0019  pred_std 0.0010
step  600  loss 0.0017  pred_std 0.0012
step  650  loss 0.0018  pred_std 0.0012
step  700  loss 0.0017  pred_std 0.0016
step  750  loss 0.0016  pred_std 0.0019
step  800  loss 0.0022  pred_std 0.0025
step  850  loss 0.0018  pred_std 0.0037
step  900  loss 0.0020  pred_std 0.0049
step  950  loss 0.0016  pred_std 0.0070
features: (8, 128)
